
# RT-DETR Inference
This notebook runs detection using Ultralytics RT-DETR.
It does not train models. It requires the VisDrone-trained weights produced by Kaggle notebook 02.


In [ ]:

import cv2
import yaml
from pathlib import Path
import sys

# Resolve the repository root regardless of where the notebook is launched,
# then expose the package source on the import path.
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(REPO_ROOT / "src"))
from aeronetra.detection.adapters import get_model_adapter
from aeronetra.counting.ops import count_vehicles
from aeronetra.counting.drawing import draw_detections, export_to_json, draw_count_summary
from aeronetra.detection.types import CountSummary
import torch


In [ ]:

# Configuration
config_path = REPO_ROOT / "configs/inference/inference.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model_name = "RT-DETR"
weights_path = REPO_ROOT / "outputs/models/rtdetr_l_visdrone_best.pt"
if not weights_path.exists():
    raise FileNotFoundError(
        f"VisDrone-trained weights not found: {weights_path}. Run the Kaggle training pipeline and download the weight file first."
    )
# VisDrone/AeroNetra class IDs: car, van, truck, tricycle, awning-tricycle, bus, motor, bicycle.
class_names = {
    0: "car", 1: "van", 2: "truck", 3: "tricycle",
    4: "awning-tricycle", 5: "bus", 6: "motor", 7: "bicycle",
}
selected_classes = list(range(8))


In [ ]:

# Load Model explicitly
try:
    print(f"Attempting to load {model_name} from {weights_path}...")
    adapter = get_model_adapter("RTDETR", weights_path, class_names, device)
    adapter.load_model()
    print("Model loaded successfully.")
except (ImportError, RuntimeError) as e:
    print(f"Failed to load RT-DETR. Error: {e}")
    adapter = None


In [ ]:

# Inference on sample image
if adapter is not None:
    sample_img_path = REPO_ROOT / "tests/fixtures/sample.jpg"
    if sample_img_path.exists():
        img = cv2.imread(str(sample_img_path))
        
        # Inference
        prediction = adapter.predict(img, conf_thresh=config["confidence_threshold"], iou_thresh=config["iou_threshold"])
        print(f"Inference time: {prediction.inference_time_ms:.2f} ms")
        
        # Filter
        prediction = prediction.filter_by_class(selected_classes)
        print(f"Detections after class filtering: {len(prediction.detections)}")
        
        # Count
        total, c_counts = count_vehicles(prediction.detections)
        summary = CountSummary("sample.jpg", total, c_counts, "RTDETR")
        
        # Draw and Save
        out_img = draw_detections(img, prediction.detections)
        out_img = draw_count_summary(out_img, summary)
        
        out_dir = REPO_ROOT / "outputs/predictions"
        out_dir.mkdir(parents=True, exist_ok=True)
        
        cv2.imwrite(str(out_dir / "rtdetr_out.jpg"), out_img)
        export_to_json(prediction.detections, out_dir / "rtdetr_preds.json")
        print("Outputs saved.")
    else:
        print(f"Sample image {sample_img_path} not found. Please add a fixture image.")
